# Images, schemas, adapters, teams, and MCP: practice

Build image/text input → request a typed image description → adapt LangChain tools → alternate researcher/evaluator turns → discover MCP fetch tools.

Use this with [the walkthrough](walkthrough.md).

**Source:**

- [5_autogen/2_lab2_autogen_agentchat.ipynb](../../../../../5_autogen/2_lab2_autogen_agentchat.ipynb)

**Run one cell at a time:** predict → run with Shift+Enter → explain → change one thing. Select the root project .venv kernel.

**Offline adaptation:** code below uses fictional data or local libraries. It never executes the linked source, reads a credential file, sends messages, starts a UI/server, or changes the course databases. Simulations are labeled; they do not verify external integrations.

## 1. Text and image inputs are different content items

A multimodal message can contain text plus an image. A structured description can constrain fields such as orientation. This local example constructs a tiny image and validates a description without a model.

**Predict:** Which orientation is valid for an image wider than it is tall?

In [ ]:
from PIL import Image
from typing import Literal
from pydantic import BaseModel
image = Image.new("RGB", (80, 40), "navy")
class Description(BaseModel):
    scene: str
    orientation: Literal["portrait", "landscape", "square"]
description = Description(scene="A fictional navy rectangle", orientation="landscape")
print(image.size, description.model_dump())
assert image.width > image.height

**Check:** landscape. This exercise does not upload an image or call a vision model.

**Change one thing:** Create a square image and update orientation. Explain that valid orientation syntax alone does not establish what a model saw.

## 2. Structured output validates shape

Pydantic models name the fields you expect and check constraints. This helps callers use results reliably, but a valid field can still contain a false claim.

**Predict:** Will confidence=2 pass the constraint?

In [ ]:
from pydantic import BaseModel, Field, ValidationError
class Report(BaseModel):
    summary: str
    confidence: float = Field(ge=0, le=1)
report = Report(summary="Fictional result", confidence=0.8)
print(report.model_dump())
try:
    Report(summary="Bad shape", confidence=2)
except ValidationError:
    print("Invalid confidence rejected")
else:
    raise AssertionError("Constraint was not enforced")

**Check:** No. The value must be between zero and one. Validation checks the constraint, not the truth of the summary.

**Change one thing:** Add a field named follow_up_questions with type list[str]. Construct a valid object using it.

## 3. A tool schema describes; a dispatcher executes

The model can request a function call. Your application decodes arguments, recognizes the name, executes the function, and returns a matching tool result. No model is needed to practice this boundary.

**Predict:** Does defining the schema call the function?

In [ ]:
import json
records = []
def record_interest(email):
    records.append(email)  # In memory only; does not send email or notifications.
    return {"recorded": True}
schema = {
    "type": "function", "function": {
        "name": "record_interest", "description": "Record a supplied address",
        "parameters": {"type": "object", "properties": {"email": {"type": "string"}},
                       "required": ["email"], "additionalProperties": False},
    },
}
assert records == []
requested = {"id": "call_1", "name": "record_interest", "arguments": '{"email":"learner@example.com"}'}
allowed = {"record_interest": record_interest}  # An explicit registry of permitted functions.
if requested["name"] not in allowed:
    raise ValueError("Unknown tool")
result = allowed[requested["name"]](**json.loads(requested["arguments"]))
tool_message = {"role": "tool", "tool_call_id": requested["id"], "content": json.dumps(result)}
print(tool_message)
assert records == ["learner@example.com"]

**Check:** No. Only the explicit dispatcher call changes records. Keep the assistant tool-call message before the matching tool result in a full conversation.

**Change one thing:** Change the requested name to an unknown function and observe the error. Then add a harmless second tool to the registry.

## 4. Team turns need a termination condition

A round-robin team alternates speakers. A marker can end the conversation; a maximum turn count stops it even if the marker never appears.

**Predict:** At what turn does this conversation stop?

In [ ]:
scripted = [("writer", "Initial draft"), ("evaluator", "Add an example"),
            ("writer", "Revised draft with an example"), ("evaluator", "APPROVE")]
seen = []
for speaker, content in scripted[:6]:
    seen.append((speaker, content))
    if "APPROVE" in content:
        break
print(seen)
assert len(seen) == 4

**Check:** Turn four. Text-based termination is a rule, not proof that the work is complete.

**Change one thing:** Remove APPROVE and add a finite fallback when no approval occurs. Discuss accidental marker mentions.

## 5. MCP tools, resources, and server lifetime

A tool performs an operation; a resource supplies readable context. A client discovers tool descriptions and calls them while the server is connected. This fixture is not a protocol transport.

**Predict:** Is listing tool names the same as executing them?

In [ ]:
from contextlib import contextmanager
calls = []
@contextmanager
def practice_server():
    tools = {"add": lambda a, b: a + b}
    try:
        calls.append("connected")
        yield tools
    finally:
        calls.append("closed")
with practice_server() as tools:
    print("Discovered:", list(tools))
    result = tools["add"](12, 30)
    print("Called add:", result)
assert result == 42 and calls == ["connected", "closed"]

**Check:** No. Discovery returns descriptions. A real stdio subprocess also has a working directory, environment, and protocol stream.

**Change one thing:** Add a read-only dictionary resource and a tool returning a fixed date. Explain how discovery differs from invocation.

## Rebuild from memory

1. Recreate the main data-flow step in a new scratch cell.
2. Add one case that should fail and identify its exception/stop condition.
3. Compare your version with the linked source cells or functions.
4. Explain where a real model or service would enter, and what it could change.

For deliberate original-source execution, see the shared [configuration guide](../../../configuration.md). All provider/transport integrations remain UNVERIFIED at runtime.